**Topic**: 
    
    Hybrid recommender system with Ollama 3.0 integration

**Dataset**: 

    Amazon appliances product review.
    N user = 1,755,732
    N item = 10,4237 with a lot of text for descriptions

**Method**: 
    
    Utilize Content-based and pyspark ALS hybrid weighted scores to generate item recommendations and use local Ollama 3.0 to get insights on product recommendations


In [1]:
#Loading data
import gzip
import pandas as pd
import json
import requests
import io


url_review = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Appliances.jsonl.gz'
url_meta = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Appliances.jsonl.gz'

response1 = requests.get(url_review, stream=True)
response2 = requests.get(url_meta, stream=True)

response1.raise_for_status()
response2.raise_for_status()

#Parse rating file:
rating = []
with gzip.GzipFile(fileobj=io.BytesIO(response1.content), mode='rb') as f:
    for line in f:
        rating.append(json.loads(line.decode('utf-8')))

#Parse item metadata file:
meta = []
with gzip.GzipFile(fileobj=io.BytesIO(response2.content), mode='rb') as f:
    for line in f:
        meta.append(json.loads(line.decode('utf-8')))

#Load into dataframes:
df_rating = pd.DataFrame(rating)
df_meta = pd.DataFrame(meta)

#Renaming and reframing the columns to be more descriptive:
df_rating.columns = ['rating', 'review_title', 'review_text', 'image', 'item_id', 'item_id_parent', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
df_meta.columns = ['main_category', 'product_name', 'average_rating', 'number_of_reviews', 'features', 'descriptions', 'price', 'images', 'videos', 'store', 'categories', 'details', 'item_id', 'bought_together', 'subtitle', 'author']

#Merging the two dataframes by the item_id:
df_com = df_rating.merge(df_meta, on='item_id', how='left')

#Check number of user and items in the dataset:
print(f'number of user: {df_com["user_id"].nunique()}')
print(f'number of item: {df_com["item_id"].nunique()}')
df_com.head(3)

number of user: 1755732
number of item: 104237


,rating,review_title,review_text,image,item_id,item_id_parent,user_id,timestamp,helpful_vote,verified_purchase,...,descriptions,price,images,videos,store,categories,details,bought_together,subtitle,author
0,5.0,Work great,work great. use a new one every month,[],B01N0TQ0OH,B01N0TQ0OH,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1519317108692,0,True,...,[],9.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Mr Coffee Water Filter Instruction...,Geesta,"[Small Appliance Parts & Accessories, Coffee &...","{'Manufacturer': 'Geesta', 'Part Number': 'Gee...",None,NaN,NaN
1,5.0,excellent product,Little on the thin side,[],B07DD2DMXB,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,1664746863446,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5.0,Happy customer!,"Quick delivery, fixed the issue!",[],B082W3Z9YK,B082W3Z9YK,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,1607225435363,0,True,...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Blutoget make life more convenient...,Romalon,"[Appliances, Parts & Accessories, Dryer Parts ...","{'Manufacturer': 'Romalon', 'Part Number': '27...",None,NaN,NaN
